In [2]:
# Ensure you have at least 2GB free space in Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())


Working in: /content/drive/MyDrive/DL_Group_MGI1


In [4]:
# Install dependencies if not already installed
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 20.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 98.3 MB/s eta 0:00:00


In [5]:
# Run this everytime you update something in the repo!
import os

REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

# if project directory is empty clone the repo, otherwise pull the latest changes
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

print("Working in:", os.getcwd())





remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 12.97 KiB | 60.00 KiB/s, done.
From https://github.com/gabrielcastrob/Deep_learning_WUR
   d8bd5f2..255bf0b  main       -> origin/main
Updating d8bd5f2..255bf0b
error: invalid object 100644 8b137891791fe96927ad78e64b0aad7bded08bdc for 'notebooks/02_ML_ResNet.ipynb'
Fast-forward
 notebooks/02_ViT_Stratified_vs_Random.ipynb | 1156 +++++++++++++++++++++++++++
 1 file changed, 1156 insertions(+)
 create mode 100644 notebooks/02_ViT_Stratified_vs_Random.ipynb
/content
Working in: /content/drive/MyDrive/DL_Group_MGI1


Download dataset

In [6]:

"Download the UCMerced Land Use dataset if not already present. "
"The dataset will be saved in the 'ucmdata' directory. "

import os
import zipfile
import subprocess
import shutil
if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')

    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
        zip_ref.extractall('UCMImages')

    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
UCM_images_path = "/ucmdata/Images/"
Multilabels_path = "/ucmdata/LandUse_Multilabeled.txt"

print(os.listdir('.'))


['.git', 'GRS_34806_MGI_Project.ipynb', 'LICENSE', 'Level2_ResNet50_UCM', 'README.md', 'notebooks', 'ucmdata', '__pycache__', 'wandb', 'outputs', 'utils.py']


Load Packages

In [7]:
import random, shutil, zipfile
from pathlib import Path
from PIL import Image
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, average_precision_score, hamming_loss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit



import torchmetrics
import re 

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Accelerator: gpu
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


Class list and dataset

Augmentation  

Data split 

In [8]:
# Full dataloader test 
# Can take a long time (4.5min on Colab GPU T4)
from utils import build_dataloaders, UCMMultilabelDataset


train_loader, val_loader, test_loader, classes, pos_w = build_dataloaders(
    root_dir = "ucmdata",
    label_file = "LandUse_Multilabeled.txt",
    image_size = (224, 224),
    batch_size = 32,
    num_workers = 2,
    val_frac = 0.15,
    test_frac = 0.15,
    seed = 42,
    image_ext = ".tif",
)

# Create a csv output of class distribution
train_labels = torch.cat([labels for _, labels in train_loader]).numpy()
val_labels = torch.cat([labels for _, labels in val_loader]).numpy()
test_labels = torch.cat([labels for _, labels in test_loader]).numpy()

print("Train class freq:", train_labels.sum(axis=0))
print("Val   class freq:", val_labels.sum(axis=0))
print("Test  class freq:", test_labels.sum(axis=0))

Train class freq: [ 70. 502. 483. 620.  81.  73.  70.  72. 682.  72. 903. 206.  70.  72.
  70. 706. 142.]
Val   class freq: [ 15. 108. 104. 133.  17.  16.  15.  16. 147.  15. 196.  44.  15.  15.
  15. 152.  31.]
Test  class freq: [ 15. 108. 104. 133.  17.  16.  15.  15. 146.  15. 201.  44.  15.  15.
  15. 151.  30.]


Initialize WANDB

In [ ]:

import wandb
wandb.login()

Hyperparameters 

In [9]:
# Define hyperparameters
PRETRAINED_MODEL = "resnet50"
NUM_CLASSES = len(classes)
MAX_EPOCHS = 25
EARLYSTOPPING_EPOCHS = 4
LR = 1e-4
WEIGHT_DECAY = 1e-5
run_count = 3


Log WandB

In [ ]:
run = wandb.init(project="multilabel-classification", # or single-label                 
                 name="ML_ptResNet50_" + str(run_count), # replace with specific name
                 config={
                     "pretrained_model": PRETRAINED_MODEL,
                     "classes": NUM_CLASSES,
                     "max_epochs": MAX_EPOCHS,
                     "learning_rate": LR,
                     "weight_decay": WEIGHT_DECAY,
                     "early_stopping_epochs": EARLYSTOPPING_EPOCHS
                     }) 


Load and train model

In [10]:
def build_resnet50(num_classes):
    weights = tvm.ResNet50_Weights.IMAGENET1K_V2
    model = tvm.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

backbone = build_resnet50(num_classes=NUM_CLASSES)
n_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"ResNet-50 trainable parameters: {n_params:,}  |  output logits: {NUM_CLASSES}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s] 

ResNet-50 trainable parameters: 23,542,865  |  output logits: 17


RESNET Head

In [ ]:
class LitResNetMultilabel(L.LightningModule):
    def __init__(self, model, num_classes, lr=1e-4, weight_decay=1e-4,
                 max_epochs=15, threshold=0.5, pos_weight=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        pos_weight = torch.as_tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None

        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.threshold = threshold

        mk = dict(task="multilabel", num_labels=num_classes, threshold=threshold)
        self.train_f1 = torchmetrics.F1Score(average="macro", **mk)
        self.val_f1   = torchmetrics.F1Score(average="macro", **mk)
        self.test_f1  = torchmetrics.F1Score(average="macro", **mk)
        self.val_acc  = torchmetrics.Accuracy(average="macro", **mk)
        self.test_acc = torchmetrics.Accuracy(average="macro", **mk)
        self.val_map  = torchmetrics.AveragePrecision(task="multilabel", num_labels=num_classes, average="macro")
        self.test_map = torchmetrics.AveragePrecision(task="multilabel", num_labels=num_classes, average="macro")


    def forward(self, x):
        return self.model(x)

    def _step(self, batch):
        imgs, labels = batch          # labels: float tensor (B, C)
        logits = self(imgs)
        loss = self.criterion(logits, labels)
        return loss, logits, labels

    def training_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.train_f1.update(logits, labels.int())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_f1",   self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.val_f1.update(logits, labels.int())
        self.val_acc.update(logits, labels.int())
        self.val_map.update(logits, labels.int())
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_f1",   self.val_f1,  on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc",  self.val_acc, on_step=False, on_epoch=True)
        self.log("val_map",  self.val_map, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        loss, logits, labels = self._step(batch)
        self.test_f1.update(logits, labels.int())
        self.test_acc.update(logits, labels.int())
        self.test_map.update(logits, labels.int())
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        self.log("test_f1",   self.test_f1,  on_step=False, on_epoch=True)
        self.log("test_acc",  self.test_acc, on_step=False, on_epoch=True)
        self.log("test_map",  self.test_map, on_step=False, on_epoch=True)

    def predict_step(self, batch, batch_idx):
        imgs, labels = batch
        logits = self(imgs)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= self.threshold).int()
        return {"probs": probs, "preds": preds, "labels": labels.int()}

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(),
                                      lr=self.hparams.lr,
                                      weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.max_epochs)
        return [optimizer], [scheduler]

Training

In [ ]:
from utils import LitPretTrainedMultilabel 

lit_model = LitPretTrainedMultilabel(model=backbone, num_classes=NUM_CLASSES,
                                lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS, pos_weight=pos_w)
# Callbacks 
checkpoint_cb = ModelCheckpoint(
    dirpath="outputs/checkpoints",
    filename="resnet50-multilabel-best-{epoch:02d}-{val_f1:.4f}",
    monitor="val_f1", mode="max", save_top_k=1, save_weights_only=True,
)

early_stopping = EarlyStopping(
            monitor="val_f1",  
            patience=EARLYSTOPPING_EPOCHS,        # Number of epochs with no improvement
            mode="max",        # Maximize F1 score
            verbose=None
            )

# Logger to plot later
csv_logger = CSVLogger(save_dir="outputs/logs", name="resnet50_multilabel")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices="auto",
    callbacks=[early_stopping, checkpoint_cb],
    logger=csv_logger,
    log_every_n_steps=10,
)
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)



Final classification on test data

In [ ]:
# Evaluate using the best checkpoint
trainer.test(lit_model, dataloaders=test_loader, ckpt_path="best")

best_path = checkpoint_cb.best_model_path
print(f"Best checkpoint: {best_path}")
lit_model = LitResNetMultilabel.load_from_checkpoint(best_path, model=lit_model.model)

# Save lit model
torch.save(lit_model.state_dict(), "outputs/resnet50_multilabel_litmodel.pth")




In [ ]:
#Load model from pth
loaded_state_dict = torch.load("outputs/resnet50_multilabel_litmodel.pth")
lit_model.load_state_dict(loaded_state_dict)


# Collect probs, preds and ground-truth for downstream analysis
preds_out = trainer.predict(lit_model, dataloaders=test_loader)
test_probs  = torch.cat([b["probs"]  for b in preds_out], dim=0).cpu().numpy()
test_preds  = torch.cat([b["preds"]  for b in preds_out], dim=0).cpu().numpy()
test_labels = torch.cat([b["labels"] for b in preds_out], dim=0).cpu().numpy()

In [ ]:
from utils import compute_test_metrics

# Usage
metrics = compute_test_metrics(test_preds, test_labels, test_probs)

# Print metrics
print(f"\nTest accuracy   : {metrics['accuracy']:.4f}")
print(f"Test macro F1   : {metrics['macro_f1']:.4f}")
print(f"Test micro F1   : {metrics['micro_f1']:.4f}")
print(f"Test samples F1 : {metrics['samples_f1']:.4f}")
print(f"Test macro mAP  : {metrics['macro_map']:.4f}")
print(f"Hamming loss    : {metrics['hamming_loss']:.4f}")
print(f"Exact-match acc : {metrics['subset_acc']:.4f}")



Log Results

In [ ]:
run.log({"macro_f1": metrics['macro_f1'], "micro_f1": metrics['micro_f1'], "samples_f1": metrics['samples_f1'],
         "macro_map": metrics['macro_map'], "hamming_loss": metrics['hamming_loss'], "subset_acc": metrics['subset_acc']})


run.finish()

Visualize

In [ ]:
metrics_csv = Path(csv_logger.log_dir) / "metrics.csv"
df = pd.read_csv(metrics_csv)
epoch_df = df.groupby("epoch").mean(numeric_only=True).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "train_loss" in epoch_df:
    axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], label="train")
if "val_loss" in epoch_df:
    axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss")
axes[0].set_title("ResNet-50 multilabel — Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

if "train_f1" in epoch_df:
    axes[1].plot(epoch_df["epoch"], epoch_df["train_f1"], label="train macro-F1")
if "val_f1" in epoch_df:
    axes[1].plot(epoch_df["epoch"], epoch_df["val_f1"], label="val macro-F1")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("macro F1")
axes[1].set_title("ResNet-50 multilabel — macro F1"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
#Path("outputs").mkdir(exist_ok=True)
#plt.savefig("outputs/resnet50_multilabel_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def plot_training_curves(csv_logger, model_name: str = "ResNet-50 multilabel", save_path: str = None):
    """
    Plot training and validation loss/F1 curves from CSV logger.
    
    Args:
        csv_logger: Lightning CSVLogger instance
        model_name: Name for the plot title
        save_path: Optional path to save the figure (e.g., "outputs/curves.png")
    
    Returns:
        tuple: (fig, axes) matplotlib objects
    """
    metrics_csv = Path(csv_logger.log_dir) / "metrics.csv"
    df = pd.read_csv(metrics_csv)
    epoch_df = df.groupby("epoch").mean(numeric_only=True).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss subplot
    if "train_loss" in epoch_df.columns:
        axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], label="train", marker="o", markersize=3)
    if "val_loss" in epoch_df.columns:
        axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], label="val", marker="s", markersize=3)
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("BCE loss")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # F1 subplot
    if "train_f1" in epoch_df.columns:
        axes[1].plot(epoch_df["epoch"], epoch_df["train_f1"], label="train macro-F1", marker="o", markersize=3)
    if "val_f1" in epoch_df.columns:
        axes[1].plot(epoch_df["epoch"], epoch_df["val_f1"], label="val macro-F1", marker="s", markersize=3)
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("macro F1")
    axes[1].set_title(f"{model_name} — macro F1")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved training curves to: {save_path}")
    
    plt.show()
    return fig, axes

In [ ]:
# Usage
from utils import plot_training_curves
plot_training_curves(csv_logger, model_name="ResNet-50 multilabel", 
                     save_path=None)

In [ ]:
from utils import plot_per_class_metrics

# Usage
fig, ax, summary_df = plot_per_class_metrics(
    test_labels=test_labels,
    test_preds=test_preds,
    test_probs=test_probs,
    classes=classes,
    macro_f1=metrics['macro_f1'],
    macro_map=metrics['macro_map'],
    model_name="ResNet-50 multilabel",
    save_path="outputs/resnet50_multilabel_per_class.png",
    csv_output="outputs/resnet50_multilabel_per_class.csv"
)

Examples of images

In [ ]:
test_ds_full = UCMMultilabelDataset(root_dir="ucmdata", label_file="LandUse_Multilabeled.txt", transform=None)
test_ds_subset = test_loader.dataset  # This is the Subset object
test_indices = test_ds_subset.indices  # Get the indices

# Map indices to image paths
test_items = [(test_ds_full.image_paths[idx], None) for idx in test_indices]


# Grid of test images with ground-truth vs predicted labels
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

N_SHOW = 9                       # 3x3 grid
COLS = 3
ROWS = (N_SHOW + COLS - 1) // COLS

# Pick a mix: a few "easy" (all correct), a few with at least one mistake.
errors_per_sample = (test_preds != test_labels).sum(axis=1)
correct_idx  = np.where(errors_per_sample == 0)[0]
wrong_idx    = np.where(errors_per_sample > 0)[0]

rng_show = np.random.default_rng(4)
n_correct = min(N_SHOW // 2, len(correct_idx))
n_wrong   = N_SHOW - n_correct
chosen = np.concatenate([
    rng_show.choice(correct_idx, n_correct, replace=False),
    rng_show.choice(wrong_idx,   n_wrong,   replace=False),
])
rng_show.shuffle(chosen)

fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 4.5, ROWS * 5.2))
axes = axes.flatten()

for ax, sample_i in zip(axes, chosen):
    rel_path, _ = test_items[sample_i]
    img = Image.open(rel_path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")

    gt   = test_labels[sample_i].astype(bool)
    pred = test_preds[sample_i].astype(bool)
    probs = test_probs[sample_i]

    tp = gt &  pred
    fp = ~gt &  pred       # predicted but not in GT
    fn = gt  & ~pred       # in GT but missed

    # Header: filename + TP/FP/FN counts
    fname = Path(rel_path).name
    ax.set_title(f"{fname}\nTP={tp.sum()}  FP={fp.sum()}  FN={fn.sum()}",
                 fontsize=10, loc="right")

    # Build a colored caption beneath the image
    lines = []
    for i in np.where(tp)[0]:
        lines.append(("green",  f"✓ {classes[i]}({probs[i]:.2f})"))
    for i in np.where(fn)[0]:
        lines.append(("red",    f"✗ {classes[i]}({probs[i]:.2f})  [missed]"))
    for i in np.where(fp)[0]:
        lines.append(("orange", f"+ {classes[i]}({probs[i]:.2f})  [extra]"))

    # Render the colored lines stacked under the image
    for j, (color, text) in enumerate(lines):
        ax.text(0.0, -0.04 - 0.06 * j, text, transform=ax.transAxes,
                fontsize=9, color=color, va="top", ha="left", family="monospace")

# Hide any unused axes
for ax in axes[len(chosen):]:
    ax.axis("off")

# Legend
legend_handles = [
    mpatches.Patch(color="green",  label="TP — correctly predicted"),
    mpatches.Patch(color="red",    label="FN — missed (in GT, not predicted)"),
    mpatches.Patch(color="orange", label="FP — extra (predicted, not in GT)"),
]
fig.legend(handles=legend_handles, loc="upper center",
           bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)

plt.tight_layout()
#Path("outputs").mkdir(exist_ok=True)
#plt.savefig("outputs/resnet50_multilabel_predictions_grid.png",
            #dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def plot_prediction_grid(test_preds, test_labels, test_probs, classes, 
                         test_loader, root_dir: str = "ucmdata", 
                         label_file: str = "LandUse_Multilabeled.txt",
                         n_show: int = 9, seed: int = 4,
                         save_path: str = None):
    """
    Plot a grid of test images with ground-truth vs predicted labels.
    
    Args:
        test_preds: Predicted labels, shape (N, C)
        test_labels: Ground truth labels, shape (N, C)
        test_probs: Predicted probabilities, shape (N, C)
        classes: List of class names
        test_loader: DataLoader for test set
        root_dir: Path to dataset root directory
        label_file: Name of label file
        n_show: Number of images to display (default 3x3 grid)
        seed: Random seed for reproducibility
        save_path: Optional path to save the figure
    
    Returns:
        fig: matplotlib figure object
    """

    
    # Load full dataset to access image paths
    test_ds_full = UCMMultilabelDataset(root_dir=root_dir, label_file=label_file, 
                                        transform=None)
    test_ds_subset = test_loader.dataset
    test_indices = test_ds_subset.indices
    test_items = [(test_ds_full.image_paths[idx], None) for idx in test_indices]
    
    # Setup grid
    cols = 3
    rows = (n_show + cols - 1) // cols
    
    # Pick a mix: correct predictions + incorrect predictions
    errors_per_sample = (test_preds != test_labels).sum(axis=1)
    correct_idx = np.where(errors_per_sample == 0)[0]
    wrong_idx = np.where(errors_per_sample > 0)[0]
    
    rng_show = np.random.default_rng(seed)
    n_correct = min(n_show // 2, len(correct_idx))
    n_wrong = n_show - n_correct
    
    chosen = np.concatenate([
        rng_show.choice(correct_idx, n_correct, replace=False),
        rng_show.choice(wrong_idx, n_wrong, replace=False),
    ])
    rng_show.shuffle(chosen)
    
    # Create figure
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 5.2))
    axes = axes.flatten()
    
    # Plot each sample
    for ax, sample_i in zip(axes, chosen):
        rel_path, _ = test_items[sample_i]
        img = Image.open(rel_path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")
        
        gt = test_labels[sample_i].astype(bool)
        pred = test_preds[sample_i].astype(bool)
        probs = test_probs[sample_i]
        
        tp = gt & pred
        fp = ~gt & pred
        fn = gt & ~pred
        
        # Title with counts
        fname = Path(rel_path).name
        ax.set_title(f"{fname}\nTP={tp.sum()}  FP={fp.sum()}  FN={fn.sum()}",
                     fontsize=10, loc="right")
        
        # Build colored label captions
        lines = []
        for i in np.where(tp)[0]:
            lines.append(("green", f"✓ {classes[i]}({probs[i]:.2f})"))
        for i in np.where(fn)[0]:
            lines.append(("red", f"✗ {classes[i]}({probs[i]:.2f})  [missed]"))
        for i in np.where(fp)[0]:
            lines.append(("orange", f"+ {classes[i]}({probs[i]:.2f})  [extra]"))
        
        # Render colored text
        for j, (color, text) in enumerate(lines):
            ax.text(0.0, -0.04 - 0.06 * j, text, transform=ax.transAxes,
                    fontsize=9, color=color, va="top", ha="left", family="monospace")
    
    # Hide unused axes
    for ax in axes[len(chosen):]:
        ax.axis("off")
    
    # Add legend
    legend_handles = [
        mpatches.Patch(color="green", label="TP — correctly predicted"),
        mpatches.Patch(color="red", label="FN — missed (in GT, not predicted)"),
        mpatches.Patch(color="orange", label="FP — extra (predicted, not in GT)"),
    ]
    fig.legend(handles=legend_handles, loc="upper center",
               bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
    
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved prediction grid to: {save_path}")
    
    plt.show()
    return fig




In [ ]:
from utils import plot_prediction_grid
fig = plot_prediction_grid(
    test_preds=test_preds,
    test_labels=test_labels,
    test_probs=test_probs,
    classes=classes,
    test_loader=test_loader,
    root_dir="ucmdata",
    label_file="LandUse_Multilabeled.txt",
    n_show=9,
    seed=4,
    save_path="outputs/resnet50_multilabel_predictions_grid.png"
)